# Lesson 5 — It Learns

No more counting. A grid of weights starts random, gets nudged by every
example, and ends up placing good bets. You will watch the loss fall —
the same number every AI lab watches.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
corpus = re.sub(r"[^a-z ]+", " ", raw.lower())
corpus = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(corpus), "characters")
print(repr(corpus[:100]))

In [ ]:
import numpy as np
np.random.seed(0)

AB = "abcdefghijklmnopqrstuvwxyz "
V = 27
to_i = {ch: i for i, ch in enumerate(AB)}

# Training data: (current letter, next letter) as numbers.
xs = np.array([to_i[corpus[i]] for i in range(len(corpus) - 1)])
ys = np.array([to_i[corpus[i + 1]] for i in range(len(corpus) - 1)])
print(len(xs), "examples")

## The model: a 27 x 27 grid of weights

Row = current letter. The row's numbers, pushed through softmax, become the
betting odds for the next letter. Right now they're random, so the odds are
garbage — let's measure exactly how much garbage.

In [ ]:
W = np.random.randn(V, V) * 0.1

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def loss():
    p = softmax(W[xs])                    # odds for every example
    return -np.log(p[np.arange(len(xs)), ys] + 1e-9).mean()

print("loss with random weights: %.3f" % loss())
print("(a model that knows nothing scores about %.3f)" % np.log(V))

## The training loop: show, score, nudge, repeat

The nudge direction comes from the difference between the model's odds and
the truth — that's the whole gradient for this model.

In [ ]:
import matplotlib.pyplot as plt

history = []
lr = 20.0
for step in range(300):
    p = softmax(W[xs])                    # current bets
    p[np.arange(len(xs)), ys] -= 1        # bets minus truth = nudge direction
    grad = np.zeros_like(W)
    np.add.at(grad, xs, p)                # collect nudges per row
    W -= lr * grad / len(xs)              # one small step
    history.append(loss())

plt.plot(history)
plt.xlabel("nudge rounds"); plt.ylabel("loss")
plt.title("The falling number that means it's working")
plt.show()
print("final loss: %.3f" % history[-1])

## Sample from the trained weights

Nothing was counted. The weights learned the same bets the counting table
had — compare this output to lesson 2's.

In [ ]:
import random

def generate(length=200, T=1.0):
    cur = to_i["t"]
    out = "t"
    for _ in range(length):
        p = softmax(W[cur] / T)
        cur = np.random.choice(V, p=p)
        out += AB[cur]
    return out

print(generate())

## Turn-in

Train twice — 50 rounds and 500 rounds (edit the loop). For each: final
loss, one generated line, one comparing sentence. Then: where does the
improvement stop, and why would it stop?

(Hint for the last question: this model still has one letter of context.
The weights can learn lesson 2's table almost perfectly — and nothing more.)